# 06 — Restaurant recommendation prototype
This is a transparent ranking baseline using available listing metadata. It is not a learned personalized recommender because the listings do not include customer interaction history.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
path = ROOT/"data/processed/restaurants_clean.csv"
restaurants = pd.read_csv(path)
display(restaurants.head())
print("Rows:", len(restaurants))

,restaurant_name,cuisine,rating,rating_count,average_price,number_of_offers,offer_name,area,pure_veg,location
0,La Pino'Z Pizza,"Pizzas, Pastas",4.0,10.0,250.0,2,FLAT DEAL\nFLAT â¹125 OFF\nUSE FLAT125ABOVE â...,LALA LAJPAT RAI MARKET,No,Abohar
1,The Second Wife,"Indian, North Indian",3.6,50.0,250.0,2,"30% OFF UPTO â¹75\nUSE TRYNEWABOVE â¹149, FL...",Central Abohar,No,Abohar
2,Tasty Bites,"Italian, Beverages",3.8,10.0,200.0,1,FLAT â¹120 OFF\nUSE AXIS120ABOVE â¹500,Central Abohar,Yes,Abohar
3,Food Studio,"Pizzas, Burgers",3.5,8.0,49.0,5,"50% OFF UPTO â¹100\nUSE TRYNEWABOVE â¹129, F...",Central Abohar,Yes,Abohar
4,Roll Express,"Fast Food, Snacks",4.3,100.0,200.0,2,DEAL OF DAY\n10% OFF UPTO â¹40\nUSE STEALDEAL...,Circular Road,No,Abohar


Rows: 139314


In [2]:
r = restaurants.copy()
r["rating"] = pd.to_numeric(r["rating"], errors="coerce")
r["rating_count"] = pd.to_numeric(r["rating_count"], errors="coerce").fillna(0)
r["average_price"] = pd.to_numeric(r["average_price"], errors="coerce")
valid_ratings = r["rating"].dropna()
global_mean = valid_ratings.mean() if len(valid_ratings) else 0
m = r["rating_count"].quantile(0.75) if len(r) else 0
if pd.isna(m): m = 0
r["ranking_score"] = (
    (r["rating_count"]/(r["rating_count"]+m))*r["rating"].fillna(global_mean)
    + (m/(r["rating_count"]+m))*global_mean
) if m > 0 else r["rating"].fillna(global_mean)
r = r.sort_values(["ranking_score", "rating_count"], ascending=False)
display(r[["restaurant_name","cuisine","rating","rating_count","average_price","area","location","ranking_score"]].head(20))

,restaurant_name,cuisine,rating,rating_count,average_price,area,location,ranking_score
9592,Om Veg Restaurant,"Maharashtrian, Healthy Food",5.0,1000.0,150.0,CIDCO,Aurangabad,4.912572
19653,Agra Chat Bhandar,"Fast Food, Street Food",5.0,1000.0,100.0,Old Bhilwara,Bhilwara,4.912572
85133,Apni Nighi Rasoi,Indian,5.0,1000.0,200.0,Model Town,Ludhiana,4.912572
22252,Cafe Frespresso,"Continental, Chinese",5.0,500.0,400.0,Patia,Bhubaneswar,4.839715
22769,Satkaar,"Indian, Chinese",5.0,500.0,250.0,Jayadev Vihar,Bhubaneswar,4.839715
46817,The Oven House,"Pizzas, Italian",5.0,500.0,250.0,Sector 37D,Gurgaon,4.839715
60608,Balaji Paratha Center,Indian,5.0,500.0,200.0,KANWAR NAGAR,Jaipur,4.839715
78655,Red Bull Baker'S,Ice Cream,5.0,500.0,200.0,Rampura,Kota,4.839715
91921,Amigos Den,"Indian, Biryani",5.0,500.0,300.0,Malad Kan East,Mumbai,4.839715
15778,Amrit'S Bikaner Mishthan Bhandar,Desserts,4.9,1000.0,200.0,Barnala,Barnala,4.821663


In [3]:
cuisine_filter = None   
location_filter = None  
max_price = None        
min_rating = None       

rec = r.copy()
if cuisine_filter:
    rec = rec[rec["cuisine"].str.contains(cuisine_filter, case=False, na=False)]
if location_filter:
    rec = rec[
        rec["location"].str.contains(location_filter, case=False, na=False)
        | rec["area"].str.contains(location_filter, case=False, na=False)
    ]
if max_price is not None:
    rec = rec[rec["average_price"].le(max_price)]
if min_rating is not None:
    rec = rec[rec["rating"].ge(min_rating)]
display(rec[["restaurant_name","cuisine","rating","rating_count","average_price","area","location","ranking_score"]].head(20))
rec.to_csv(ROOT/"reports/restaurant_ranking.csv", index=False)

,restaurant_name,cuisine,rating,rating_count,average_price,area,location,ranking_score
9592,Om Veg Restaurant,"Maharashtrian, Healthy Food",5.0,1000.0,150.0,CIDCO,Aurangabad,4.912572
19653,Agra Chat Bhandar,"Fast Food, Street Food",5.0,1000.0,100.0,Old Bhilwara,Bhilwara,4.912572
85133,Apni Nighi Rasoi,Indian,5.0,1000.0,200.0,Model Town,Ludhiana,4.912572
22252,Cafe Frespresso,"Continental, Chinese",5.0,500.0,400.0,Patia,Bhubaneswar,4.839715
22769,Satkaar,"Indian, Chinese",5.0,500.0,250.0,Jayadev Vihar,Bhubaneswar,4.839715
46817,The Oven House,"Pizzas, Italian",5.0,500.0,250.0,Sector 37D,Gurgaon,4.839715
60608,Balaji Paratha Center,Indian,5.0,500.0,200.0,KANWAR NAGAR,Jaipur,4.839715
78655,Red Bull Baker'S,Ice Cream,5.0,500.0,200.0,Rampura,Kota,4.839715
91921,Amigos Den,"Indian, Biryani",5.0,500.0,300.0,Malad Kan East,Mumbai,4.839715
15778,Amrit'S Bikaner Mishthan Bhandar,Desserts,4.9,1000.0,200.0,Barnala,Barnala,4.821663
